# Skin Lesion Classification — Task 01
This notebook reproduces the comparison tables required by **Task_01.docx**, following the
transfer-learning + dilated-convolution methodology described in *"Skin Lesions Classification
Using Deep Learning Based on Dilated Convolution"* (Ratul et al.), adapted to the Kaggle
**Skin Cancer: 9 Classes (ISIC)** dataset.

It produces three results tables:

1. **Table 1 — Transfer Learning Models**: AlexNet, VGG16, VGG19, ResNet18, ResNet50,
   ResNet101, DenseNet121, EfficientNet-B0 — each fine-tuned end-to-end and evaluated with
   Accuracy / Precision / Recall / F1 / AUC.
2. **Table 2 — Deep Features + Classical Classifiers**: features extracted from the
   best-performing CNN backbone, then fed to Logistic Regression, Decision Tree, Random
   Forest, KNN, Linear SVM, RBF-SVM, and XGBoost.
3. **Table 3 — Computational Efficiency**: Parameters, model size, FLOPs, inference time,
   and accuracy for every architecture in Table 1.

At the end, the notebook writes all results directly into a copy of `Task_01.docx`.

> Run on a GPU runtime (Runtime → Change runtime type → GPU) for reasonable training times.


## 1. Install dependencies

In [1]:
!pip -q install kagglehub torchmetrics thop python-docx xgboost scikit-learn --upgrade


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 121.3 MB/s eta 0:00:00


## 2. Download the dataset from Kaggle

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Path to dataset files: /kaggle/input/skin-cancer9-classesisic


In [3]:
import os

# Inspect the downloaded directory structure so we can locate the train/test folders
# regardless of the exact nesting Kaggle gives us.
for root, dirs, files in os.walk(path):
    depth = root.replace(path, "").count(os.sep)
    if depth <= 2:
        indent = "  " * depth
        print(f"{indent}{os.path.basename(root) or root}/")
        if depth == 2:
            print("  " * (depth + 1) + f"... {len(files)} files")


skin-cancer9-classesisic/
  Skin cancer ISIC The International Skin Imaging Collaboration/
    Test/
      ... 0 files
    Train/
      ... 0 files


In [4]:
import os

def find_dir(root, keyword):
    """Walk the dataset tree and return the first directory whose name contains `keyword`."""
    keyword = keyword.lower()
    for dirpath, dirnames, _ in os.walk(root):
        for d in dirnames:
            if keyword in d.lower():
                return os.path.join(dirpath, d)
    return None

train_dir = find_dir(path, "train")
test_dir = find_dir(path, "test")

assert train_dir is not None, "Could not locate a Train directory — inspect the tree above and set train_dir manually."
assert test_dir is not None, "Could not locate a Test directory — inspect the tree above and set test_dir manually."

print("Train dir:", train_dir)
print("Test dir :", test_dir)

CLASS_NAMES = sorted(os.listdir(train_dir))
NUM_CLASSES = len(CLASS_NAMES)
print(f"Found {NUM_CLASSES} classes: {CLASS_NAMES}")


Train dir: /kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Train
Test dir : /kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Test
Found 9 classes: ['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']


## 3. Data loading & preprocessing
We hold out 15% of the training split as a validation set (stratified by class via
`ImageFolder` + `random_split` per class is approximated with a simple random split, which is
adequate given the dataset is reasonably balanced across the 9 classes).

In [5]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

IMG_SIZE = 224
BATCH_SIZE = 32
VAL_FRACTION = 0.15
SEED = 42

torch.manual_seed(SEED)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomAffine(degrees=0, translate=(0.2, 0.2), shear=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_train_ds = datasets.ImageFolder(train_dir, transform=train_transform)
test_ds = datasets.ImageFolder(test_dir, transform=eval_transform)

n_val = int(len(full_train_ds) * VAL_FRACTION)
n_train = len(full_train_ds) - n_val
train_ds, val_ds = random_split(full_train_ds, [n_train, n_val],
                                 generator=torch.Generator().manual_seed(SEED))
# validation set should use eval-time transforms (no augmentation) — patch in place
val_ds.dataset = datasets.ImageFolder(train_dir, transform=eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Train: 1904 | Val: 335 | Test: 118
Device: cuda


## 4. Model zoo — the 8 transfer-learning architectures
Each model is loaded with ImageNet-pretrained weights and its final classification layer is
replaced to output `NUM_CLASSES` logits, mirroring the paper's transfer-learning + fine-tuning
strategy (Section IV-E of the paper).

In [6]:
import torch.nn as nn
from torchvision import models

def build_model(name, num_classes):
    """Return a torchvision model with its classifier head replaced for num_classes."""
    name = name.lower()

    if name == "alexnet":
        m = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)

    elif name == "vgg16":
        m = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)

    elif name == "vgg19":
        m = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)

    elif name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, num_classes)

    elif name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, num_classes)

    elif name == "resnet101":
        m = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, num_classes)

    elif name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)

    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)

    else:
        raise ValueError(f"Unknown architecture: {name}")

    return m

ARCHITECTURES = [
    "alexnet", "vgg16", "vgg19",
    "resnet18", "resnet50", "resnet101",
    "densenet121", "efficientnet_b0",
]

# Display names used when we write results into the Word tables
DISPLAY_NAME = {
    "alexnet": "AlexNet",
    "vgg16": "VGG16",
    "vgg19": "VGG19",
    "resnet18": "ResNet18",
    "resnet50": "ResNet50",
    "resnet101": "ResNet101",
    "densenet121": "DenseNet121",
    "efficientnet_b0": "EfficientNet-B0",
}


## 5. Generic training loop

In [7]:
import time
import copy
import torch.optim as optim
from tqdm.auto import tqdm

def train_model(model, train_loader, val_loader, num_epochs=5, lr=1e-4, freeze_epochs=3):
    """
    Two-stage fine-tuning, mirroring the paper (Section IV-E):
      1) freeze the backbone for `freeze_epochs`, train only the new classifier head.
      2) unfreeze everything and continue training end-to-end.
    Returns the best model (by validation accuracy) and its training history.
    """
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()

    # ---- stage 1: freeze backbone -----------------------------------------
    for p in model.parameters():
        p.requires_grad = False
    for p in _classifier_params(model):
        p.requires_grad = True

    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    best_val_acc, best_state = 0.0, copy.deepcopy(model.state_dict())
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(num_epochs):
        if epoch == freeze_epochs:
            # ---- stage 2: unfreeze everything ---------------------------
            for p in model.parameters():
                p.requires_grad = True
            optimizer = optim.Adam(model.parameters(), lr=lr * 0.1)

        model.train()
        running_loss = 0.0
        for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
        train_loss = running_loss / len(train_loader.dataset)

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        val_loss /= total
        val_acc = correct / total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"  epoch {epoch+1}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model, history


def _classifier_params(model):
    """Return the parameters of just the newly-attached classifier head."""
    if hasattr(model, "fc"):
        return model.fc.parameters()
    if hasattr(model, "classifier"):
        return model.classifier.parameters()
    raise ValueError("Model has neither .fc nor .classifier")


## 6. Evaluation — Accuracy / Precision / Recall / F1 / AUC

In [8]:
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score)
from sklearn.preprocessing import label_binarize

@torch.no_grad()
def evaluate_model(model, loader, num_classes):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)
        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)
    y_true_bin = label_binarize(y_true, classes=list(range(num_classes)))

    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred) * 100,
        "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0) * 100,
        "Recall": recall_score(y_true, y_pred, average="macro", zero_division=0) * 100,
        "F1-Score": f1_score(y_true, y_pred, average="macro", zero_division=0) * 100,
        "AUC": roc_auc_score(y_true_bin, y_prob, average="macro", multi_class="ovr") * 100,
    }
    return metrics, y_true, y_pred, y_prob


## 7. Train & evaluate all 8 architectures (Table 1)
This is the most time-consuming cell — expect it to take a while even on a Colab GPU.
Reduce `NUM_EPOCHS` for a quicker smoke test.

In [9]:
NUM_EPOCHS = 5  # paper uses 200; lower this for practical Colab runtimes

table1_results = {}
trained_models = {}
histories = {}

for arch in ARCHITECTURES:
    print(f"\n=== Training {DISPLAY_NAME[arch]} ===")
    model = build_model(arch, NUM_CLASSES)
    model, history = train_model(model, train_loader, val_loader, num_epochs=NUM_EPOCHS)
    metrics, y_true, y_pred, y_prob = evaluate_model(model, test_loader, NUM_CLASSES)
    print(f"  Test metrics: {metrics}")

    table1_results[arch] = metrics
    trained_models[arch] = model.cpu()  # move off GPU to free memory between runs
    histories[arch] = history
    torch.cuda.empty_cache()



=== Training AlexNet ===
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 174MB/s]


Epoch 1/5:   0%|          | 0/60 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79ef648ae700>
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79ef648ae700>
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
self._shutdown_workers()    
if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    assert self._parent_pid == os.getpid(), 'can only test a child process'    
if w.is_alive():AssertionError
:   File "/usr/lib/python3.13/multiprocessing/pro

  epoch 1: train_loss=1.6205  val_loss=1.3037  val_acc=0.5463


...

  epoch 3: train_loss=1.8364  val_loss=1.8293  val_acc=0.4119


Epoch 4/5:   0%|          | 0/60 [00:00<?, ?it/s]

  epoch 4: train_loss=1.7713  val_loss=1.7727  val_acc=0.4358


Epoch 5/5:   0%|          | 0/60 [00:00<?, ?it/s]

  epoch 5: train_loss=1.7087  val_loss=1.7079  val_acc=0.4537
  Test metrics: {'Accuracy': 29.66101694915254, 'Precision': 23.11985976671885, 'Recall': 30.32407407407407, 'F1-Score': 23.13161040617981, 'AUC': 77.82058113100312}


In [10]:
import pandas as pd

table1_df = pd.DataFrame({
    DISPLAY_NAME[arch]: table1_results[arch] for arch in ARCHITECTURES
}).T
table1_df = table1_df[["Accuracy", "Precision", "Recall", "F1-Score", "AUC"]].round(2)
table1_df.index.name = "Model"
print("Table 1 — Comparison of Transfer Learning Models")
table1_df


Table 1 — Comparison of Transfer Learning Models


,Accuracy,Precision,Recall,F1-Score,AUC
Model,,,,,
AlexNet,49.15,46.34,49.31,41.76,88.52
VGG16,43.22,40.80,44.44,37.45,85.71
VGG19,51.69,51.24,51.39,46.65,88.55
ResNet18,37.29,27.36,39.58,28.22,83.72
ResNet50,38.98,29.13,40.97,29.43,85.34
ResNet101,44.07,48.83,45.14,36.96,85.71
DenseNet121,34.75,31.82,34.49,28.21,81.62
EfficientNet-B0,29.66,23.12,30.32,23.13,77.82


## 8. Table 2 — Deep Features + Classical Classifiers
We pick the **best-performing backbone from Table 1** (by test accuracy) as the fixed deep
feature extractor, remove its classification head, and feed the resulting feature vectors into
seven classical ML classifiers.

In [11]:
best_arch = max(table1_results, key=lambda a: table1_results[a]["Accuracy"])
print(f"Best backbone for feature extraction: {DISPLAY_NAME[best_arch]}")

feature_model = trained_models[best_arch].to(DEVICE)
feature_model.eval()

def get_feature_extractor(model, arch):
    """Strip the classification head so forward() returns the penultimate feature vector."""
    model = copy.deepcopy(model)
    if hasattr(model, "fc"):
        model.fc = nn.Identity()
    elif hasattr(model, "classifier"):
        if isinstance(model.classifier, nn.Sequential):
            model.classifier = nn.Sequential(*list(model.classifier.children())[:-1])
        else:
            model.classifier = nn.Identity()
    return model

extractor = get_feature_extractor(feature_model, best_arch).to(DEVICE).eval()

@torch.no_grad()
def extract_features(loader, extractor):
    feats, labels = [], []
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE)
        f = extractor(imgs)
        f = torch.flatten(f, 1).cpu().numpy()
        feats.append(f)
        labels.append(lbls.numpy())
    return np.concatenate(feats), np.concatenate(labels)

X_train, y_train = extract_features(train_loader, extractor)
X_test, y_test = extract_features(test_loader, extractor)
print("Feature shapes:", X_train.shape, X_test.shape)


Best backbone for feature extraction: VGG19
Feature shapes: (1904, 4096) (118, 4096)


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

CLASSIFIERS = {
    # REMOVED multi_class="auto" to fix the TypeError
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=SEED),
    "Decision Tree": DecisionTreeClassifier(random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=SEED),
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(n_neighbors=5),
    "Linear SVM": SVC(kernel="linear", probability=True, random_state=SEED),
    "RBF-SVM": SVC(kernel="rbf", probability=True, random_state=SEED),
    # REMOVED deprecated use_label_encoder=False from XGBoost to prevent warnings
    "XGBoost": XGBClassifier(n_estimators=300, eval_metric="mlogloss", random_state=SEED),
}

table2_results = {}
for clf_name, clf in CLASSIFIERS.items():
    print(f"Training {clf_name} ...")
    clf.fit(X_train_s, y_train)
    y_pred = clf.predict(X_test_s)
    y_prob = clf.predict_proba(X_test_s)
    y_true_bin = label_binarize(y_test, classes=list(range(NUM_CLASSES)))

    table2_results[clf_name] = {
        "Accuracy": accuracy_score(y_test, y_pred) * 100,
        "Precision": precision_score(y_test, y_pred, average="macro", zero_division=0) * 100,
        "Recall": recall_score(y_test, y_pred, average="macro", zero_division=0) * 100,
        "F1-Score": f1_score(y_test, y_pred, average="macro", zero_division=0) * 100,
        "AUC": roc_auc_score(y_true_bin, y_prob, average="macro", multi_class="ovr") * 100,
    }

table2_df = pd.DataFrame(table2_results).T
table2_df = table2_df[["Accuracy", "Precision", "Recall", "F1-Score", "AUC"]].round(2)
table2_df.index.name = "Classifier"
table2_df.insert(0, "Feature Extractor", f"Deep Features ({DISPLAY_NAME[best_arch]})")
print("Table 2 — Comparison of Different Classifiers")
table2_df

Training Logistic Regression ...
Training Decision Tree ...
Training Random Forest ...
Training K-Nearest Neighbors (KNN) ...
Training Linear SVM ...


/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Training RBF-SVM ...


/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Training XGBoost ...
Table 2 — Comparison of Different Classifiers


,Feature Extractor,Accuracy,Precision,Recall,F1-Score,AUC
Classifier,,,,,,
Logistic Regression,Deep Features (VGG19),44.07,53.20,48.15,46.31,80.19
Decision Tree,Deep Features (VGG19),33.90,28.39,36.81,29.16,64.20
Random Forest,Deep Features (VGG19),48.31,50.55,48.61,42.53,88.40
K-Nearest Neighbors (KNN),Deep Features (VGG19),44.92,48.58,45.83,43.99,77.44
Linear SVM,Deep Features (VGG19),44.07,53.30,48.15,45.82,86.83
RBF-SVM,Deep Features (VGG19),36.44,42.46,32.87,30.51,82.04
XGBoost,Deep Features (VGG19),45.76,43.44,46.53,41.22,88.49


## 9. Table 3 — Computational Efficiency
For every architecture in Table 1 we measure parameter count, on-disk model size, FLOPs
(via `thop`), and average single-image inference latency.

In [14]:
from thop import profile

def measure_efficiency(model, img_size=224, n_runs=50):
    model = model.to(DEVICE).eval()
    dummy = torch.randn(1, 3, img_size, img_size).to(DEVICE)

    # Parameters
    n_params = sum(p.numel() for p in model.parameters()) / 1e6  # millions

    # FLOPs
    flops, _ = profile(model, inputs=(dummy,), verbose=False)
    flops_g = flops / 1e9  # GFLOPs

    # Model size on disk (state_dict, float32)
    buffer_path = "/tmp/_tmp_model.pt"
    torch.save(model.state_dict(), buffer_path)
    import os as _os
    size_mb = _os.path.getsize(buffer_path) / 1e6
    _os.remove(buffer_path)

    # Inference time (average over n_runs, after a short warm-up)
    with torch.no_grad():
        for _ in range(10):
            _ = model(dummy)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.time()
        for _ in range(n_runs):
            _ = model(dummy)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        elapsed_ms = (time.time() - start) / n_runs * 1000

    return {
        "Parameters (M)": round(n_params, 2),
        "Model Size (MB)": round(size_mb, 2),
        "FLOPs (G)": round(flops_g, 2),
        "Inference Time (ms)": round(elapsed_ms, 2),
    }

table3_results = {}
for arch in ARCHITECTURES:
    print(f"Measuring {DISPLAY_NAME[arch]} ...")
    eff = measure_efficiency(trained_models[arch])
    eff["Accuracy (%)"] = round(table1_results[arch]["Accuracy"], 2)
    table3_results[arch] = eff

table3_df = pd.DataFrame({
    DISPLAY_NAME[arch]: table3_results[arch] for arch in ARCHITECTURES
}).T
table3_df.index.name = "Model"
print("Table 3 — Computational Efficiency Comparison")
table3_df


Measuring AlexNet ...
Measuring VGG16 ...
Measuring VGG19 ...
Measuring ResNet18 ...
Measuring ResNet50 ...
Measuring ResNet101 ...
Measuring DenseNet121 ...
Measuring EfficientNet-B0 ...
Table 3 — Computational Efficiency Comparison


,Parameters (M),Model Size (MB),FLOPs (G),Inference Time (ms),Accuracy (%)
Model,,,,,
AlexNet,57.04,228.17,0.71,2.10,49.15
VGG16,134.30,537.20,15.47,10.81,43.22
VGG19,139.61,558.44,19.63,12.33,51.69
ResNet18,11.18,44.81,1.82,2.98,37.29
ResNet50,23.53,94.43,4.13,8.15,38.98
ResNet101,42.52,170.72,7.86,11.41,44.07
DenseNet121,6.96,28.50,2.90,15.82,34.75
EfficientNet-B0,4.02,16.48,0.41,8.27,29.66


## 10. Write the results into `Task_01.docx`
Upload the original `Task_01.docx` (the one containing the three empty tables) using the file
picker below, then run this cell — it fills every "—" placeholder in order and saves
`Task_01_filled.docx`, which is downloaded automatically.

In [15]:
from google.colab import files

print("Please upload Task_01.docx")
uploaded = files.upload()
docx_path = next(iter(uploaded.keys()))


Please upload Task_01.docx


Saving Task 01.docx to Task 01.docx


In [16]:
from docx import Document

doc = Document(docx_path)

def unique_row_cells(row):
    """
    python-docx repeats the same Cell object for horizontally-merged cells when you
    iterate row.cells (Table 2 in this document merges some header/label columns).
    This returns only the logically distinct cells, in order, so indexing lines up
    with the DataFrame columns instead of the raw (merged) grid columns.
    """
    cells, seen_ids = [], []
    for cell in row.cells:
        if id(cell._tc) not in seen_ids:
            cells.append(cell)
            seen_ids.append(id(cell._tc))
    return cells

def fill_table_by_label(table, label_to_values, label_col=0, value_start_col=1):
    """
    Fills a table by matching each data row's label cell (e.g. the model or
    classifier name already printed in the template) against the keys of
    `label_to_values`, rather than assuming positional order. This is important
    here because Table 3 in this template omits ResNet101 even though Table 1
    includes it — matching by label keeps every row correct regardless of which
    subset of rows a given table happens to contain.

    label_to_values: dict mapping the row's visible label (case-insensitive,
                      whitespace-stripped) -> list of values to write, in column order.
    """
    normalized = {k.strip().lower(): v for k, v in label_to_values.items()}
    for row in table.rows[1:]:  # skip header row
        cells = unique_row_cells(row)
        label = cells[label_col].text.strip().lower()
        if label not in normalized:
            print(f"  [warn] no results found for row labelled {cells[label_col].text!r} — left as-is")
            continue
        for c_offset, val in enumerate(normalized[label]):
            cells[value_start_col + c_offset].text = str(val)

METRIC_ORDER = ["Accuracy", "Precision", "Recall", "F1-Score", "AUC"]

# --- Table 1: Transfer Learning Models --------------------------------------
t1_by_label = {
    DISPLAY_NAME[arch]: [f"{table1_results[arch][m]:.2f}" for m in METRIC_ORDER]
    for arch in ARCHITECTURES
}
fill_table_by_label(doc.tables[0], t1_by_label, label_col=0, value_start_col=1)

# --- Table 2: Deep Features + Classifiers -----------------------------------
t2_by_label = {
    name: [f"{table2_results[name][m]:.2f}" for m in METRIC_ORDER]
    for name in CLASSIFIERS.keys()
}
# cols 0,1 are Feature Extractor / Classifier labels; match on the Classifier column (col 1)
fill_table_by_label(doc.tables[1], t2_by_label, label_col=1, value_start_col=2)

# --- Table 3: Computational Efficiency (note: this template omits ResNet101) ---
t3_by_label = {
    DISPLAY_NAME[arch]: [
        table3_results[arch]["Parameters (M)"], table3_results[arch]["Model Size (MB)"],
        table3_results[arch]["FLOPs (G)"], table3_results[arch]["Inference Time (ms)"],
        table3_results[arch]["Accuracy (%)"],
    ]
    for arch in ARCHITECTURES
}
fill_table_by_label(doc.tables[2], t3_by_label, label_col=0, value_start_col=1)

out_path = "Task_01_filled.docx"
doc.save(out_path)
files.download(out_path)
print("Saved and downloading:", out_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved and downloading: Task_01_filled.docx
